# Step 3b: LightGBM LambdaRank (final model)

**Pipeline.** `step1_data_preprocess.ipynb` → `step2_baselines.ipynb` → `step3_lightgbm.ipynb` (base model) → `step3_lambdarank.ipynb` (this notebook, final model).

This notebook asks one question: **does switching the loss from pointwise BCE to listwise LambdaRank improve the ranking?** Everything else is held fixed so that the loss is the only difference from `step3_lightgbm.ipynb`. LambdaRank has the highest validation Hit@10 (our KPI) of all models, so it is our final model (picked 8 Nov).

| # | Decision | Choice |
|---|---|---|
| D22 | Model | LightGBM, `objective="lambdarank"`. Only change from the base model's `PARAMS` |
| D23 | Held fixed | Same 65 features, same binary target (engaged = click, cart or purchase), same `fit` training rows (D18), same hand-set `PARAMS` (D20), same early stopping on validation NDCG@10 (D19), same user split, metric code and candidate slates |
| D24 | Comparison | Against LightGBM BCE on the **same validation rows**, using its saved scores (`lgbm_scores_val.parquet`). Paired over users: bootstrap 95% CI of the per-user difference, plus McNemar (binary metrics) and Wilcoxon signed-rank (graded metrics) p-values |
| D25 | Test | Scored once, in the final run (`RUN_TEST = True`), after the final model is picked on validation. Test scores are saved in the same format as validation scores, so a blend picked on validation can be applied to test |
| D26 | Frozen model | The 8 Nov run (`BT4222_step3_lambdarank_val_08Nov_1356`, validation Hit@10 0.7231). LambdaRank does not reproduce exactly when retrained (an identical earlier run gave 0.7213), so later runs **load this saved model** (`LOAD_MODEL_FROM`) instead of retraining |

What changes inside LightGBM: with BCE, each row is scored on its own and `group` is only used by the NDCG stopping metric. With LambdaRank, `group` defines the slates, and the gradient for each product comes from pairs (engaged, not engaged) **within its slate**, weighted by how much swapping the pair would change NDCG. So the training rows must be contiguous by slate (checked below).

## How to run in Colab
1. Sign in with an account that can see the shared `BT4222_artifacts` folder.
2. Runtime → Run all (CPU is enough). Keep all outputs when you save.
3. `LOAD_MODEL_FROM` names the frozen run's results folder: the notebook loads `lgbm_lambdarank.txt` from it instead of retraining, so the test run scores exactly the model picked on validation. Whoever runs it needs that folder shared with them. Set it to `""` to train a new model.
4. The last cell uploads the results to a new folder named in `RESULT_FOLDER` (date and time in the name, so runs never overwrite each other). Never upload to `BT4222_artifacts`. Post the folder name and the GROUP CHAT line (Section 3) in the group chat.

## 1. Setup

In [ ]:
import os, sys, json, time, gc
import numpy as np
import pandas as pd
import lightgbm as lgb

IN_COLAB = "google.colab" in sys.modules

# ---- config: change these, nothing else ----
ART_DIR = os.environ.get("JD_ART_DIR", "/content/artifacts" if IN_COLAB else ".")    # step 1 and step 2 files
OUT_DIR = os.environ.get("JD_OUT_DIR", "/content/results" if IN_COLAB else "results") # result tables and the model
DRIVE_ARTIFACT_FOLDER = "BT4222_artifacts"     # shared folder with step 1 and 2 files (read by fixed id, see _folder_id)
SEED = 42
NDCG_GAIN = "linear"                           # must match step 2: "linear" (gain = label) or "exp" (2^label - 1)
RUN_TEST = True                               # True only for the final run: test is scored once
# -------------------------------------------
# new Drive folder for this run's results: never BT4222_artifacts, never reused (date and time in the name)
RESULT_FOLDER = f"BT4222_step3_lambdarank_{'test' if RUN_TEST else 'val'}_" + pd.Timestamp.now(tz="Asia/Singapore").strftime("%d%b_%H%M")
EXPECTED_VAL = {"hit@10": 0.7231, "ndcg@10": 0.4168}   # validation figures the model was frozen with; a re-run must match
LOAD_MODEL_FROM = "BT4222_step3_lambdarank_val_08Nov_1356"   # frozen run (8 Nov): load its saved model instead of retraining. "" = train a new model
GROUP_FOLDER_ID = "1YpCRQtsc8dqPi6Q4E9yTfc0wOkpCJ3Z8"     # BT4222-Group-15, where the frozen run folders are
assert RESULT_FOLDER != DRIVE_ARTIFACT_FOLDER, "never write to the shared artifacts folder"
os.makedirs(OUT_DIR, exist_ok=True)
print("results will upload to My Drive/" + RESULT_FOLDER)

def _drive():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    return build("drive", "v3")

def _folder_id(svc):
    return "1Yb4AGglIJV4Pdd8Sx6NozG8Klh6uweqH"

def drive_download(names):
    # fetch the named files from My Drive/DRIVE_ARTIFACT_FOLDER into ART_DIR, skipping ones already there
    import io
    from googleapiclient.http import MediaIoBaseDownload
    if not names:
        return
    svc = _drive(); fid = _folder_id(svc); os.makedirs(ART_DIR, exist_ok=True)
    for f in names:
        found = svc.files().list(q=f"'{fid}' in parents and name='{f}' and trashed=false", fields="files(id)").execute()["files"]
        if not found:
            raise FileNotFoundError(f"{f} is not in My Drive/{DRIVE_ARTIFACT_FOLDER}. Run the step that makes it first.")
        with io.FileIO(f"{ART_DIR}/{f}", "wb") as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=found[0]["id"]), chunksize=100 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
        print("downloaded", f)

def load_frozen_model(fname):
    """Load the frozen model file fname from the Drive folder LOAD_MODEL_FROM (yours, or shared with you)."""
    path = f"{OUT_DIR}/{fname}"
    if not os.path.exists(path):
        import io
        from googleapiclient.http import MediaIoBaseDownload
        svc = _drive()
        q = f"name='{LOAD_MODEL_FROM}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        # look inside the group folder first, by its id: a name search misses folders you reach only through the group folder
        folders = (svc.files().list(q=f"'{GROUP_FOLDER_ID}' in parents and {q}", fields="files(id)").execute()["files"]
                   or svc.files().list(q=q, fields="files(id)").execute()["files"])
        hits = []
        for fo in folders:
            hits += svc.files().list(q=f"'{fo['id']}' in parents and name='{fname}' and trashed=false",
                                     fields="files(id)").execute()["files"]
        if len(hits) != 1:
            raise FileNotFoundError(f"expected one {fname} in a Drive folder named {LOAD_MODEL_FROM}, found {len(hits)}. "
                                    "Ask the person who ran it to share that folder with you.")
        with io.FileIO(path, "wb") as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=hits[0]["id"])); done = False
            while not done:
                _, done = dl.next_chunk()
    m = lgb.Booster(model_file=path)
    m.best_iteration = m.current_iteration()   # the saved file holds only the trees up to the frozen run's best iteration
    print(f"loaded frozen model {fname} from My Drive/{LOAD_MODEL_FROM}: {m.current_iteration()} trees, no retraining")
    return m

# Shared metric definitions (src/metrics.py). On Colab without the repo, an identical copy is defined here.
try:
    from src.metrics import MetricAccumulator, bootstrap_ci
except ImportError:
    def _rank(scores, rng):
        # descending score, ties broken uniformly at random (never by stored order: the raw file stores engaged items first)
        return np.lexsort((rng.random(len(scores)), -np.asarray(scores, float)))
    def _metrics(ranked, k, gain):
        g = ranked.astype(float) if gain == "linear" else np.power(2.0, ranked) - 1.0
        rel = np.flatnonzero(ranked > 0)
        disc = 1.0 / np.log2(np.arange(2, k + 2))
        dcg = (g[:k] * disc[: len(g[:k])]).sum()
        ideal = np.sort(g)[::-1][:k]
        idcg = (ideal * disc[: len(ideal)]).sum()
        return {f"hit@{k}": float((ranked[:k] > 0).any()), f"ndcg@{k}": float(dcg / idcg) if idcg > 0 else 0.0,
                "mrr": float(1.0 / (rel[0] + 1)) if len(rel) else 0.0, "p@1": float(ranked[0] > 0)}
    class MetricAccumulator:
        def __init__(self, k=10, gain="linear", seed=42):
            self.k, self.gain, self.rng, self.rows = k, gain, np.random.default_rng(seed), []
        def add(self, scores, labels):
            labels = np.asarray(labels)
            if not (labels > 0).any():
                return None
            m = _metrics(labels[_rank(scores, self.rng)], self.k, self.gain); self.rows.append(m); return m
        def per_slate(self):
            return {key: np.array([r[key] for r in self.rows]) for key in self.rows[0]} if self.rows else {}
    def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
        values = np.asarray(list(values), float); rng = np.random.default_rng(seed)
        means = values[rng.integers(0, len(values), size=(n_boot, len(values)))].mean(axis=1)
        return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

METRICS = ["hit@10", "ndcg@10", "mrr", "p@1"]

def evaluate(df, score):
    """Rank each slate in df (rows grouped by user) by score; one row of metrics per slate."""
    users = df.user.values
    assert (np.diff(users) >= 0).all(), "rows must be grouped by user"
    starts = np.r_[0, np.flatnonzero(np.diff(users)) + 1]; ends = np.r_[starts[1:], len(users)]
    s, y = np.asarray(score, float), df.label.values
    acc, kept = MetricAccumulator(gain=NDCG_GAIN, seed=SEED), []
    for a, b in zip(starts, ends):
        if acc.add(s[a:b], y[a:b]) is not None:
            kept.append(users[a])
    out = pd.DataFrame(acc.per_slate()); out.insert(0, "user", kept)
    return out

def summarise(per, name):
    row = {"model": name, "slates": len(per)}
    row.update({m: per[m].mean() for m in METRICS})
    for m in ["hit@10", "ndcg@10"]:
        lo, hi = bootstrap_ci(per[m]); row[f"{m} 95% CI"] = f"{lo:.3f} to {hi:.3f}"
    return row

FMT = {m: "{:.4f}" for m in METRICS} | {"slates": "{:,}"}
BASELINE_NAMES = {"score_random": "Random", "score_query": "Query only", "score_pop": "Popularity",
                  "score_repeat_pop": "Repeat → popularity", "score_mf": "MF (truncated SVD)",
                  "score_t3_maxprob": "Team 3 NCF (as published: max class prob.)",
                  "score_t3_pengaged": "Team 3 NCF (fixed: P(engaged))"}
OURS, REF = "LightGBM LambdaRank (ours)", "LightGBM BCE (reference)"

results will upload to My Drive/BT4222_step3_lambdarank_test_08Nov_1704


In [ ]:
if IN_COLAB and DRIVE_ARTIFACT_FOLDER:
    drive_download(["features.parquet", "users.parquet", "config.json", "baseline_scores.parquet", "lgbm_scores_val.parquet"])
cfg = json.load(open(f"{ART_DIR}/config.json"))
FEATURES = cfg["FEATURES"]
FEATURES = [c for c in FEATURES if c != "slate_len"]
CATEGORICAL = ["item_last_type", "cand_cate1"]
K = pd.read_parquet(f"{ART_DIR}/features.parquet",
                    columns=sorted(set(["user", "wid", "label", "split", "fit", "train_eval", "pop_w0", "pop_w1", "pop_w2"] + FEATURES)))
U = pd.read_parquet(f"{ART_DIR}/users.parquet")
B = pd.read_parquet(f"{ART_DIR}/baseline_scores.parquet")
BASE_COLS = [c for c in B.columns if c.startswith("score_")]
K["cand_cate1"] = pd.Categorical(K.cand_cate1)
K["item_last_type"] = K.item_last_type.astype("category")

LABEL_DERIVED = {"label", "n_pos", "split", "fit", "train_eval", "t3_list"}
assert not LABEL_DERIVED & set(FEATURES), "a label-derived column is in the feature list"
print(f"lightgbm {lgb.__version__} | {len(FEATURES)} features | step 1: MAX_USERS={cfg['MAX_USERS']}, HISTORY_CAP={cfg['HISTORY_CAP']}")
print("baselines from step 2:", [BASELINE_NAMES.get(c, c) for c in BASE_COLS])

assert cfg["MAX_USERS"] == 0 and len(U) == 171728 and len(K) == 6690773 and len(B) == 3966935, "not the full step 1 and 2 output"
assert "user_hist_len" in U.columns, "the segment cell needs user_hist_len"
assert len(FEATURES) == 65, "the feature list must stay at the locked 65"

# LightGBM BCE's validation scores, saved by step3_lightgbm.ipynb: the reference for the paired comparison
R = pd.read_parquet(f"{ART_DIR}/lgbm_scores_val.parquet")
REF_COL = [c for c in R.columns if c.startswith("score_")]
assert len(REF_COL) == 1, f"expected one score column in lgbm_scores_val.parquet, found {REF_COL}"
R = R.rename(columns={REF_COL[0]: "score_bce", "label": "label_ref"})[["user", "wid", "label_ref", "score_bce"]]
print(f"reference scores: {len(R):,} validation rows, {R.user.nunique():,} users")

downloaded features.parquet
downloaded users.parquet
downloaded config.json
downloaded baseline_scores.parquet
downloaded lgbm_scores_val.parquet
lightgbm 4.6.0 | 65 features | step 1: MAX_USERS=0, HISTORY_CAP=None
baselines from step 2: ['Random', 'Query only', 'Popularity', 'Repeat → popularity', 'MF (truncated SVD)', 'Team 3 NCF (as published: max class prob.)', 'Team 3 NCF (fixed: P(engaged))']
reference scores: 1,538,177 validation rows, 17,172 users


## 2. Train

In [ ]:
TR = K[K.fit.values]                                         # sampled train rows, grouped by user
VA = K[K.split.values == 1].reset_index(drop=True)            # full validation slates
y_tr = (TR.label.values > 0).astype(int)

# LambdaRank reads slates from `group`, which assumes each user's rows are contiguous and in the same order as the data.
# (For BCE this only affected the stopping metric; here it defines the training pairs.)
def check_contiguous(df, name):
    u = df.user.values
    n_runs = 1 + int((u[1:] != u[:-1]).sum())
    assert n_runs == df.user.nunique(), f"{name}: a user's rows are split into several blocks, so group sizes would be wrong"
check_contiguous(TR, "train"); check_contiguous(VA, "validation")
g_tr = TR.groupby("user", sort=False).size()
assert (g_tr.index.values == pd.unique(TR.user.values)).all()
print(f"train slates {len(g_tr):,} | rows per slate: median {int(g_tr.median())}, max {g_tr.max()} | "
      f"slates with no positive (no LambdaRank gradient): {(TR.groupby('user', sort=False).label.max() == 0).mean():.1%}")
print(f"train rows {len(TR):,} (positive rate {y_tr.mean():.1%}) | validation rows {len(VA):,} in {VA.user.nunique():,} slates")

# group = slate sizes. LambdaRank forms its (engaged, not engaged) pairs within each group
dtrain = lgb.Dataset(TR[FEATURES], y_tr, group=g_tr.values,
                     categorical_feature=CATEGORICAL, free_raw_data=False)
dval = lgb.Dataset(VA[FEATURES], (VA.label.values > 0).astype(int), group=VA.groupby("user", sort=False).size().values,
                   categorical_feature=CATEGORICAL, reference=dtrain)
# identical to step3_lightgbm.ipynb except objective="lambdarank" (binary labels: gain 1 for engaged, 0 otherwise)
PARAMS = dict(objective="lambdarank", metric="ndcg", eval_at=[10], learning_rate=0.05, num_leaves=63,
              min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, seed=SEED, verbose=-1)
if LOAD_MODEL_FROM:
    model = load_frozen_model("lgbm_lambdarank.txt")      # test scores exactly the model picked on validation
else:
    t0 = time.time()
    model = lgb.train(PARAMS, dtrain, num_boost_round=3000, valid_sets=[dval], valid_names=["val"],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(100)])
    print(f"best iteration {model.best_iteration} | {time.time() - t0:.0f}s")
model.save_model(f"{OUT_DIR}/lgbm_lambdarank.txt")

train slates 137,380 | rows per slate: median 21, max 72 | slates with no positive (no LambdaRank gradient): 0.0%
train rows 2,937,620 (positive rate r8.7%) | validation rows 1,538,177 in 17,172 slates


loaded frozen model lgbm_lambdarank.txt from My Drive/BT4222_step3_lambdarank_val_08Nov_1356: 517 trees, no retraining


## 3. Results

**RESULT: validation.** Every model is scored by the same code on the same validation slates: LambdaRank, the BCE reference (its saved scores, re-scored here) and every step 2 baseline. LambdaRank scores are not probabilities, only their order within a slate matters, which is all the metrics use.

In [ ]:
def with_baselines(split_code):
    """Our model's scores for one split, joined to step 2's baseline scores on (user, product)."""
    part = K[K.split.values == split_code].reset_index(drop=True)
    part["score_lambdarank"] = model.predict(part[FEATURES], num_iteration=model.best_iteration)
    part = part.merge(B[["user", "wid"] + BASE_COLS], on=["user", "wid"], how="left", validate="one_to_one")
    assert part[BASE_COLS].notna().all().all(), "step 2 scores are missing for some rows: re-run step 2 on this step 1 output"
    return part

def results(part):
    per = {BASELINE_NAMES.get(c, c): evaluate(part, part[c]) for c in BASE_COLS}
    if "score_bce" in part:
        per[REF] = evaluate(part, part.score_bce)
    per[OURS] = evaluate(part, part.score_lambdarank)
    return per, pd.DataFrame([summarise(p, n) for n, p in per.items()]).set_index("model")

PV = with_baselines(1)
PV = PV.merge(R, on=["user", "wid"], how="left", validate="one_to_one")   # left merge keeps PV's row order
assert len(R) == len(PV) and PV.score_bce.notna().all(), "lgbm_scores_val.parquet does not cover the same validation rows"
assert (PV.label.values == PV.label_ref.values).all(), "labels differ from the reference file: not the same target or rows"
per_val, res_val = results(PV)
display(res_val.style.format(FMT))
res_val.to_csv(f"{OUT_DIR}/step3_lambdarank_val_results.csv")

TE = K[K.train_eval.values].reset_index(drop=True)
TE["score_lambdarank"] = model.predict(TE[FEATURES], num_iteration=model.best_iteration)
tr_row = summarise(evaluate(TE, TE.score_lambdarank), OURS)
print("TRAIN (10,000 users, full slates):", {m: round(tr_row[m], 4) for m in METRICS})
print("VAL                              :", {m: round(res_val.loc[OURS, m], 4) for m in METRICS})

# sanity check: the reference re-scored here should match step3_lightgbm (Hit@10 0.7188, NDCG@10 0.4146)
for m, v in [("hit@10", 0.7188), ("ndcg@10", 0.4146)]:
    got = res_val.loc[REF, m]
    print(f"BCE reference {m}: {got:.4f} (step3_lightgbm {v})" + ("" if abs(got - v) < 0.0015 else "  <-- MISMATCH: check the reference file"))

# a re-run (e.g. the final RUN_TEST run) retrains LambdaRank: it must reproduce the frozen validation figures
for m, v in EXPECTED_VAL.items():
    got = res_val.loc[OURS, m]
    print(f"LambdaRank val {m}: {got:.4f} (frozen {v})" + ("" if abs(got - v) < 0.0015 else "  <-- MISMATCH: do not report this run's test numbers"))

# long table (model, split, metrics) for the group results table. Baselines and BCE come from step3_lightgbm's summary
SUMMARY = pd.DataFrame([{"model": OURS, "split": "train", **{k: v for k, v in tr_row.items() if k != "model"}},
                        {"model": OURS, "split": "val", **res_val.loc[OURS].to_dict()}])

,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"17,172",0.2240,0.0826,0.1008,0.0302,0.218 to 0.230,0.080 to 0.085
Query only,"17,172",0.2349,0.0912,0.1085,0.0368,0.228 to 0.241,0.088 to 0.094
Popularity,"17,172",0.6437,0.3448,0.3388,0.1972,0.637 to 0.651,0.340 to 0.350
Repeat → popularity,"17,172",0.6813,0.3856,0.3869,0.2463,0.675 to 0.689,0.381 to 0.391
MF (truncated SVD),"17,172",0.6029,0.3196,0.3181,0.1837,0.596 to 0.610,0.315 to 0.324
Team 3 NCF (as published: max class prob.),"17,172",0.2533,0.1079,0.1214,0.0477,0.247 to 0.260,0.105 to 0.111
Team 3 NCF (fixed: P(engaged)),"17,172",0.4507,0.2187,0.2244,0.1165,0.443 to 0.458,0.214 to 0.223
LightGBM BCE (reference),"17,172",0.7188,0.4146,0.4155,0.2717,0.712 to 0.726,0.409 to 0.420
LightGBM LambdaRank (ours),"17,172",0.7231,0.4168,0.4169,0.2735,0.717 to 0.730,0.412 to 0.422


TRAIN (10,000 users, full slates): {'hit@10': np.float64(0.7415), 'ndcg@10': np.float64(0.4379), 'mrr': np.float64(0.4408), 'p@1': np.float64(0.2986)}
VAL                              : {'hit@10': np.float64(0.7231), 'ndcg@10': np.float64(0.4168), 'mrr': np.float64(0.4169), 'p@1': np.float64(0.2735)}
BCE reference hit@10: 0.7188 (step3_lightgbm 0.7188)
BCE reference ndcg@10: 0.4146 (step3_lightgbm 0.4146)
LambdaRank val hit@10: 0.7231 (frozen 0.7231)
LambdaRank val ndcg@10: 0.4168 (frozen 0.4168)


### Is LambdaRank better than BCE? (paired, validation)

Both models are scored on the same validation slates, so the comparison is paired by user: for each user, LambdaRank's metric minus BCE's.
- **Bootstrap 95% CI** of the mean difference (same method as the rest of the project). If it excludes 0, the difference is not luck.
- **p-value**: McNemar's exact test for the 0/1 metrics (Hit@10, P@1), which only uses users where the two models disagree; Wilcoxon signed-rank for NDCG@10 and MRR.

In [ ]:
from scipy import stats

d = per_val[OURS].merge(per_val[REF], on="user", suffixes=("", "_ref"), validate="one_to_one")
assert len(d) == len(per_val[OURS]) == len(per_val[REF]), "the two models were not scored on the same slates"

def paired_row(m):
    a, b = d[m].values, d[f"{m}_ref"].values
    diff = a - b; lo, hi = bootstrap_ci(diff)
    if m in ("hit@10", "p@1"):                       # 0/1 per user: McNemar exact on the discordant users
        win, loss = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
        p = stats.binomtest(win, win + loss, 0.5).pvalue if win + loss else 1.0
        test, detail = "McNemar", f"{win:,} users only LambdaRank hits, {loss:,} only BCE"
    else:
        p = stats.wilcoxon(diff).pvalue if (diff != 0).any() else 1.0
        test, detail = "Wilcoxon", f"{(diff > 0).sum():,} users better, {(diff < 0).sum():,} worse"
    return {"metric": m, "LambdaRank": a.mean(), "BCE": b.mean(), "diff": diff.mean(),
            "95% CI": f"{lo:+.4f} to {hi:+.4f}", "test": test, "p": p, "detail": detail, "_lo": lo, "_hi": hi}

P = pd.DataFrame([paired_row(m) for m in METRICS]).set_index("metric")
display(P.drop(columns=["_lo", "_hi"]).style.format({"LambdaRank": "{:.4f}", "BCE": "{:.4f}", "diff": "{:+.4f}", "p": "{:.3g}"}))
P.drop(columns=["_lo", "_hi"]).to_csv(f"{OUT_DIR}/step3_lambdarank_vs_bce_val.csv")

for m in ["hit@10", "ndcg@10"]:
    lo, hi = P.loc[m, "_lo"], P.loc[m, "_hi"]
    verdict = "LambdaRank better" if lo > 0 else "BCE better" if hi < 0 else "no significant difference"
    print(f"{m}: {verdict} (diff {P.loc[m, 'diff']:+.4f}, CI {P.loc[m, '95% CI']}, p = {P.loc[m, 'p']:.3g})")

# one line to paste into the group chat
r = res_val.loc[OURS]
print(f"\nGROUP CHAT: LambdaRank val ({int(r.slates):,} slates): Hit@10 {r['hit@10']:.4f}, NDCG@10 {r['ndcg@10']:.4f}, "
      f"MRR {r['mrr']:.4f}, P@1 {r['p@1']:.4f} | vs BCE: Hit@10 {P.loc['hit@10', 'diff']:+.4f} (CI {P.loc['hit@10', '95% CI']}), "
      f"NDCG@10 {P.loc['ndcg@10', 'diff']:+.4f} (CI {P.loc['ndcg@10', '95% CI']}) | best iter {model.best_iteration} "
      f"| results in {RESULT_FOLDER}")

,LambdaRank,BCE,diff,95% CI,test,p,detail
metric,,,,,,,
hit@10,0.7231,0.7188,+0.0043,+0.0012 to +0.0073,McNemar,0.00813,"407 users only LambdaRank hits, 334 only BCE"
ndcg@10,0.4168,0.4146,+0.0022,+0.0007 to +0.0037,Wilcoxon,0.00592,"3,224 users better, 3,052 worse"
mrr,0.4169,0.4155,+0.0014,-0.0006 to +0.0033,Wilcoxon,0.005,"4,502 users better, 4,228 worse"
p@1,0.2735,0.2717,+0.0017,-0.0015 to +0.0047,McNemar,0.318,"436 users only LambdaRank hits, 406 only BCE"


hit@10: LambdaRank better (diff +0.0043, CI +0.0012 to +0.0073, p = 0.00813)
ndcg@10: LambdaRank better (diff +0.0022, CI +0.0007 to +0.0037, p = 0.00592)

GROUP CHAT: LambdaRank val (17,172 slates): Hit@10 0.7231, NDCG@10 0.4168, MRR 0.4169, P@1 0.2735 | vs BCE: Hit@10 +0.0043 (CI +0.0012 to +0.0073), NDCG@10 +0.0022 (CI +0.0007 to +0.0037) | best iter 517 | results in BT4222_step3_lambdarank_test_08Nov_1704


### Gain over the best baseline

Same check as the base notebook, for LambdaRank: per-user difference against the best step 2 baseline, bootstrapped over users.

In [ ]:
best = res_val.drop([OURS, REF])["ndcg@10"].idxmax()
db = per_val[OURS].merge(per_val[best], on="user", suffixes=("", "_base"))
rows = []
for m in METRICS:
    diff = db[m] - db[f"{m}_base"]; lo, hi = bootstrap_ci(diff)
    rows.append({"metric": m, "ours": db[m].mean(), best: db[f"{m}_base"].mean(), "gain": diff.mean(), "95% CI": f"{lo:+.4f} to {hi:+.4f}"})
print("best baseline on validation NDCG@10:", best)
display(pd.DataFrame(rows).set_index("metric").style.format({"ours": "{:.4f}", best: "{:.4f}", "gain": "{:+.4f}"}))

best baseline on validation NDCG@10: Repeat → popularity


,ours,Repeat → popularity,gain,95% CI
metric,,,,
hit@10,0.7231,0.6813,+0.0418,+0.0367 to +0.0468
ndcg@10,0.4168,0.3856,+0.0311,+0.0285 to +0.0338
mrr,0.4169,0.3869,+0.0300,+0.0269 to +0.0333
p@1,0.2735,0.2463,+0.0271,+0.0223 to +0.0320


## 4. What the model relies on

Gain share per feature, next to the BCE model's shares from its notebook where reported. A big shift would mean the listwise loss uses the features differently, not just reweights the same signal.

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=FEATURES)
imp = (imp / imp.sum()).sort_values(ascending=False)
display(imp.head(15).to_frame("share of total gain").style.format("{:.1%}"))
print("features never used:", imp[imp == 0].index.tolist())
print("BCE model (step3_lightgbm findings): pop_impr_train_slate_pct 27.1%, pop_w0_slate_pct 24.8%, item_days_since 8.1%, "
      "shop_share_w0_slate_pct 7.2%, cand_cate1 6.4%")

,share of total gain
pop_w0_slate_pct,30.8%
pop_impr_train_slate_pct,28.6%
item_days_since,10.4%
cand_cate1,4.3%
pop_w1,3.9%
name_overlap_slate_pct,2.7%
shop_share_w0_slate_pct,2.7%
item_last_type,2.1%
shop_days_since,2.1%
pop_impr_train,1.7%


features never used: ['cand_has_meta', 'item_flw', 'query_seen_before']
BCE model (step3_lightgbm findings): pop_impr_train_slate_pct 27.1%, pop_w0_slate_pct 24.8%, item_days_since 8.1%, shop_share_w0_slate_pct 7.2%, cand_cate1 6.4%


## 5. Where LambdaRank differs from BCE (segments)

The same validation users split into groups, comparing LambdaRank with the BCE reference inside each group. Small groups have wide CIs: only read a difference as real if its CI excludes 0.
- **History length**: light users (<5 actions), 5 to 100, 100+.
- **Engaged product**: "all cold" means no engaged product in the slate was touched in any train user's history.
- **Engaged products in slate**: LambdaRank's pairs come from slates with engagement; slates with several engaged products give it more pairs.

In [ ]:
seg = d.copy()
hl = U.set_index("user").user_hist_len.reindex(seg.user).values
seg["history length"] = pd.cut(hl, [-1, 4, 100, 10**9], labels=["<5 actions", "5-100", "100+"])
pos = PV[PV.label.values > 0]
cold = (pos.pop_w0 + pos.pop_w1 + pos.pop_w2).values == 0
seg["engaged product"] = pd.Series(cold).groupby(pos.user.values).all().reindex(seg.user).map({True: "all cold", False: "some warm"}).values
n_pos = pos.groupby("user").size().reindex(seg.user).values
seg["engaged products in slate"] = pd.cut(n_pos, [0, 1, 3, 10**9], labels=["1", "2-3", "4+"])

def seg_table(col, m="ndcg@10"):
    out = []
    for g, s in seg.groupby(col, observed=True):
        diff = s[m] - s[f"{m}_ref"]; lo, hi = bootstrap_ci(diff)
        out.append({col: g, "slates": len(s), "LambdaRank": s[m].mean(), "BCE": s[f"{m}_ref"].mean(),
                    "diff": diff.mean(), "95% CI": f"{lo:+.3f} to {hi:+.3f}"})
    return pd.DataFrame(out).set_index(col)

for col in ["history length", "engaged product", "engaged products in slate"]:
    print(f"NDCG@10 by {col}")
    display(seg_table(col).style.format({"LambdaRank": "{:.4f}", "BCE": "{:.4f}", "diff": "{:+.4f}"}))

NDCG@10 by history length


,slates,LambdaRank,BCE,diff,95% CI
history length,,,,,
<5 actions,766,0.3837,0.3871,-0.0034,-0.010 to +0.003
5-100,9098,0.4111,0.4083,+0.0028,+0.001 to +0.005
100+,7308,0.4273,0.4253,+0.0020,-0.000 to +0.004


NDCG@10 by engaged product


,slates,LambdaRank,BCE,diff,95% CI
engaged product,,,,,
all cold,3117,0.1973,0.1979,-0.0006,-0.004 to +0.003
some warm,14055,0.4654,0.4627,+0.0028,+0.001 to +0.005


NDCG@10 by engaged products in slate


,slates,LambdaRank,BCE,diff,95% CI
engaged products in slate,,,,,
1,10347,0.4612,0.4592,+0.0020,-0.000 to +0.004
2-3,5053,0.3804,0.3774,+0.0030,+0.001 to +0.005
4+,1772,0.2611,0.2605,+0.0006,-0.003 to +0.004


## 6. Test (final run only) and saved scores

Test runs only when `RUN_TEST = True`, after the group has picked the final model on validation. Validation scores are always saved with columns `user, wid, split, label, score_lambdarank`, so every model can be compared on the same rows.

In [ ]:
if RUN_TEST:
    PT = with_baselines(2)
    _, res_test = results(PT)
    display(res_test.style.format(FMT))
    res_test.to_csv(f"{OUT_DIR}/step3_lambdarank_test_results.csv")
    # test scores in the same format as the validation file, so a blend picked on validation can be applied to test
    PT[["user", "wid", "split", "label", "score_lambdarank"]].to_parquet(f"{OUT_DIR}/lambdarank_scores_test.parquet", index=False)
    SUMMARY = pd.concat([SUMMARY, pd.DataFrame([{"model": OURS, "split": "test", **res_test.loc[OURS].to_dict()}])], ignore_index=True)
else:
    print("RUN_TEST is False: test users not scored.")
PV[["user", "wid", "split", "label", "score_lambdarank"]].to_parquet(f"{OUT_DIR}/lambdarank_scores_val.parquet", index=False)
SUMMARY.to_csv(f"{OUT_DIR}/step3_lambdarank_summary.csv", index=False)
print(SUMMARY[["model", "split", "slates", "hit@10", "ndcg@10", "mrr", "p@1"]].round(4).to_string(index=False))
print("saved", sorted(os.listdir(OUT_DIR)))


,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"17,176",0.2220,0.0812,0.0991,0.0296,0.215 to 0.228,0.078 to 0.084
Query only,"17,176",0.2388,0.0931,0.1081,0.0367,0.232 to 0.245,0.090 to 0.096
Popularity,"17,176",0.6343,0.3395,0.3362,0.1982,0.627 to 0.642,0.335 to 0.345
Repeat → popularity,"17,176",0.6762,0.3840,0.3875,0.2488,0.669 to 0.683,0.379 to 0.390
MF (truncated SVD),"17,176",0.6017,0.3191,0.3210,0.1888,0.595 to 0.609,0.314 to 0.324
Team 3 NCF (as published: max class prob.),"17,176",0.2538,0.1053,0.1189,0.0454,0.247 to 0.260,0.102 to 0.109
Team 3 NCF (fixed: P(engaged)),"17,176",0.4427,0.2134,0.2188,0.1124,0.436 to 0.449,0.209 to 0.218
LightGBM LambdaRank (ours),"17,176",0.7190,0.4146,0.4145,0.2707,0.713 to 0.726,0.409 to 0.420


                     model split  slates  hit@10  ndcg@10    mrr    p@1
LightGBM LambdaRank (ours) train   10000  0.7415   0.4379 0.4408 0.2986
LightGBM LambdaRank (ours)   val   17172  0.7231   0.4168 0.4169 0.2735
LightGBM LambdaRank (ours)  test   17176  0.7190   0.4146 0.4145 0.2707
saved ['lambdarank_scores_test.parquet', 'lambdarank_scores_val.parquet', 'lgbm_lambdarank.txt', 'step3_lambdarank_summary.csv', 'step3_lambdarank_test_results.csv', 'step3_lambdarank_val_results.csv', 'step3_lambdarank_vs_bce_val.csv']


In [ ]:
# KEY RESULTS: the final model (LambdaRank) against LightGBM BCE and the best baseline. Test rows appear only when RUN_TEST = True
KEY = ["hit@10", "ndcg@10", "mrr", "p@1"]
kr_rows = [{"model": OURS, "split": "train", "slates": tr_row["slates"], **{m: tr_row[m] for m in KEY}}]
for split, res in [("val", res_val)] + ([("test", res_test)] if RUN_TEST else []):
    for name in [OURS, REF, best]:
        if name in res.index:          # BCE test scores come from step3_lightgbm, not this notebook
            kr_rows.append({"model": name, "split": split, **res.loc[name, ["slates"] + KEY].to_dict()})
key_results = pd.DataFrame(kr_rows); key_results["slates"] = key_results.slates.astype(int)
key_results.to_csv(f"{OUT_DIR}/step3_lambdarank_key_results.csv", index=False)
print(f"KEY RESULTS (full slates; best baseline = {best}, picked on validation NDCG@10)")
print(key_results.round(4).to_string(index=False))
print("\nLambdaRank minus BCE, validation (paired over users):")
print(P.loc[["hit@10", "ndcg@10"], ["LambdaRank", "BCE", "diff", "95% CI", "p"]].round(4).to_string())

KEY RESULTS (full slates; best baseline = Repeat → popularity, picked on validation NDCG@10)
                     model split  slates  hit@10  ndcg@10    mrr    p@1
LightGBM LambdaRank (ours) train   10000  0.7415   0.4379 0.4408 0.2986
LightGBM LambdaRank (ours)   val   17172  0.7231   0.4168 0.4169 0.2735
  LightGBM BCE (reference)   val   17172  0.7188   0.4146 0.4155 0.2717
       Repeat → popularity   val   17172  0.6813   0.3856 0.3869 0.2463
LightGBM LambdaRank (ours)  test   17176  0.7190   0.4146 0.4145 0.2707
       Repeat → popularity  test   17176  0.6762   0.3840 0.3875 0.2488

LambdaRank minus BCE, validation (paired over users):
         LambdaRank     BCE    diff              95% CI       p
metric                                                         
hit@10       0.7231  0.7188  0.0043  +0.0012 to +0.0073  0.0081
ndcg@10      0.4168  0.4146  0.0022  +0.0007 to +0.0037  0.0059


In [ ]:
from googleapiclient.http import MediaFileUpload
assert RESULT_FOLDER != DRIVE_ARTIFACT_FOLDER   # set in Section 1: a new folder for every run
svc = _drive()
fid = svc.files().create(body={"name": RESULT_FOLDER, "mimeType": "application/vnd.google-apps.folder"}, fields="id").execute()["id"]
for f in sorted(os.listdir(OUT_DIR)):
    media = MediaFileUpload(f"{OUT_DIR}/{f}", resumable=True, chunksize=100 * 1024 * 1024)
    svc.files().create(body={"name": f, "parents": [fid]}, media_body=media).execute()
    print("uploaded", f)
print(f"GROUP CHAT: step3_lambdarank results in My Drive/{RESULT_FOLDER} (RUN_TEST = {RUN_TEST})")

uploaded lambdarank_scores_test.parquet
uploaded lambdarank_scores_val.parquet
uploaded lgbm_lambdarank.txt
uploaded step3_lambdarank_key_results.csv
uploaded step3_lambdarank_summary.csv
uploaded step3_lambdarank_test_results.csv
uploaded step3_lambdarank_val_results.csv
uploaded step3_lambdarank_vs_bce_val.csv
GROUP CHAT: step3_lambdarank results in My Drive/BT4222_step3_lambdarank_test_08Nov_1704 (RUN_TEST = True)


## Findings (full data, full slates, frozen 8 Nov model; validation unless marked Test)
- **LambdaRank vs BCE: higher Hit@10 and NDCG@10 in this run.** Hit@10 0.7231 vs 0.7188: +0.0043 (95% CI +0.0012 to +0.0073; McNemar p = 0.008). Only 741 of 17,172 users change Hit@10 between the two models: 407 hit only with LambdaRank, 334 only with BCE. NDCG@10 0.4168 vs 0.4146: +0.0022 (CI +0.0007 to +0.0037; Wilcoxon p = 0.006). MRR (+0.0014, CI −0.0006 to +0.0033) and P@1 (+0.0017, CI −0.0015 to +0.0047) tie. MRR's Wilcoxon p (0.005) is below 0.05 although its CI includes 0: Wilcoxon tests whether more users improve than worsen (4,502 vs 4,228), not the mean. We go by the CI.
- **The Hit@10 gain is not robust across runs.** An earlier run with identical code, data and settings stopped at iteration 716 (this one at 517) and scored Hit@10 0.7213: +0.0025 over BCE (CI −0.0005 to +0.0055), not proven. The two runs match for the first 400 rounds, then drift apart (multi-threaded arithmetic; BCE reproduces exactly). So run-to-run variation (about 0.002) is about the size of the gain itself. The NDCG@10 gain was proven in both runs (+0.0021 and +0.0022). We report this run because it has the complete record (current code and checks, saved model and scores), and every later run loads its saved model (D26).
- **It beats the best baseline by more than BCE does.** Against repeat → popularity: Hit@10 +0.042 (CI +0.037 to +0.047), NDCG@10 +0.031 (CI +0.029 to +0.034); BCE's gains are +0.038 and +0.029.
- **A little more overfitting than BCE.** Train Hit@10 0.742 vs validation 0.723 (gap 0.018; BCE 0.015). NDCG@10 0.438 vs 0.417 (gap 0.021; BCE 0.013). The parameters are BCE's, not tuned for LambdaRank (D23).
- **The gain comes from mid-history users and warm slates.** NDCG@10: users with 5–100 actions +0.0028 (CI +0.001 to +0.005; 9,098 slates); slates with at least one warm engaged product +0.0028 (CI +0.001 to +0.005; 14,055); slates with 2–3 engaged products +0.0030 (CI +0.001 to +0.005; 5,053). No proven difference for light users (<5 actions, −0.0034, CI −0.010 to +0.003), heavy users (100+, +0.0020, CI −0.000 to +0.004), all-cold slates (−0.0006) or slates with 1 or 4+ engaged products.
- **Same features, shifted further toward popularity and repeats.** `pop_w0_slate_pct` 30.8% of gain (BCE 24.8%), `pop_impr_train_slate_pct` 28.6% (27.1%), `item_days_since` 10.4% (8.1%). The shop-affinity percentile drops from 7.2% to 2.7%. The same three features are never used: `cand_has_meta`, `item_flw`, `query_seen_before`.
- **Output is a ranking score, not a probability.** Only the order within a slate is meaningful. Reporting engagement probabilities, as the interim report promised, would need a calibration step.
- **Test (17,176 users, scored once on 8 Nov with the saved model):** Hit@10 **0.7190**, NDCG@10 **0.4146**, MRR 0.4145, P@1 0.2707 (validation 0.7231, 0.4168, 0.4169, 0.2735). Against BCE on test (paired means, `step4`): Hit@10 +0.0012, NDCG@10 +0.0025, MRR +0.0019, P@1 +0.0018. The Hit@10 lead is a quarter of the validation lead (+0.0043), in line with the run-to-run variation above; paired test CIs are in `step4_blend_and_segments.ipynb`. Against repeat → popularity on test: Hit@10 +0.043, NDCG@10 +0.031 (BCE: +0.042, +0.028). Train–test gap: Hit@10 0.023, NDCG@10 0.023 (BCE 0.016, 0.015).
- **Paired test CI (step 4, 17,176 users):** the NDCG@10 gain over BCE is proven (95% CI +0.0010 to +0.0040), but the Hit@10 gain is not (CI −0.0019 to +0.0041); the Hit@10 gain over repeat → popularity is proven (CI +0.0381 to +0.0479).
